## Data processing: Extract 2023 House Index (IV) from aedic indices

This notebook reads `Índices_aédicos_Oasis-La Quinia_y_otros barrio_2022-2024.xlsx` and extracts the 2023 "IV" (House index) per month for the following:

- OASIS (sheet `OASIS`)
- LA QUININA (sheet `LA QUININA`)
- ONDAS DEL CARIBE, PANTANO, 8 DE DICIEMBRE (sheet `OASIS-B` in multi-block format)
- LA COQUERA, SAN JACINTO, NUEVA BETHEL (sheet `LA QUININA-B` in multi-block format)

Output is a single DataFrame `df_iv_2023` with columns: `Neighborhood`, `Mes`, `IV`.


In [13]:
import pandas as pd
import numpy as np
import re
from typing import Dict, List, Tuple, Optional

excel_path = "/home/pescuder/repast.hpc/dengue-habm-santamarta/Índices_aédicos_Oasis-La Quinia_y_otros barrio_2022-2024.xlsx"

# Utilities

def normalize_header(value: str) -> str:
    if value is None:
        return ""
    v = str(value).strip()
    v = re.sub(r"\s+", " ", v)
    return v

MONTH_ALIASES = {
    # Spanish month mappings and common variants
    "enero": "Enero", "ene": "Enero",
    "febrero": "Febrero", "feb": "Febrero",
    "marzo": "Marzo", "mar": "Marzo",
    "abril": "Abril", "abr": "Abril",
    "mayo": "Mayo",
    "junio": "Junio", "jun": "Junio",
    "julio": "Julio", "jul": "Julio",
    "agosto": "Agosto", "ago": "Agosto",
    "septiembre": "Septiembre", "setiembre": "Septiembre", "sep": "Septiembre",
    "octubre": "Octubre", "oct": "Octubre",
    "noviembre": "Noviembre", "nov": "Noviembre",
    "diciembre": "Diciembre", "dic": "Diciembre",
}

MONTH_ORDER = {m: i for i, m in enumerate([
    "Enero","Febrero","Marzo","Abril","Mayo","Junio","Julio","Agosto","Septiembre","Octubre","Noviembre","Diciembre"
], start=1)}


def normalize_month(value: str) -> Optional[str]:
    if value is None:
        return None
    s = str(value).strip()
    if s == "":
        return None
    key = re.sub(r"[^A-Za-zÁÉÍÓÚáéíóúñÑ]", "", s).lower()
    return MONTH_ALIASES.get(key, s)


def find_year_cells(df: pd.DataFrame, year: str = "2023") -> List[Tuple[int, int]]:
    """Return (row, col) indices where a cell contains 'Año 2023'."""
    hits: List[Tuple[int,int]] = []
    pattern = re.compile(rf"a[ñn]o\s*{re.escape(year)}", re.IGNORECASE)
    for r in range(len(df)):
        row = df.iloc[r]
        for c, val in enumerate(row):
            if pd.isna(val):
                continue
            if pattern.search(str(val)):
                hits.append((r, c))
    return hits


def _dedupe_columns(cols: List[Optional[str]]) -> List[str]:
    seen: Dict[str, int] = {}
    out: List[str] = []
    for col in cols:
        name = (col or "").strip()
        if name == "":
            name = "__empty__"
        cnt = seen.get(name, 0)
        if cnt == 0:
            out.append(name)
        else:
            out.append(f"{name}__{cnt+1}")
        seen[name] = cnt + 1
    return out


def extract_table_below(df: pd.DataFrame, start_row: int, expected_cols: List[str], anchor_col: Optional[int] = None) -> pd.DataFrame:
    """Given a starting row (title like 'Año 2023'), find header in the next few lines,
    then read rows until a blank gap. If anchor_col is provided, prefer columns near that index."""
    # search header within next 5 rows
    header_row = None
    for r in range(start_row + 1, min(start_row + 6, len(df))):
        row = [normalize_header(x) for x in df.iloc[r].tolist()]
        joined = "|".join(row).lower()
        if any(ec.lower() in joined for ec in expected_cols):
            header_row = r
            break
    if header_row is None:
        return pd.DataFrame(columns=["Mes","IV"])

    # Build (deduped) header names
    raw_headers = [normalize_header(v) for v in df.iloc[header_row].tolist()]
    headers = _dedupe_columns(raw_headers)

    # Collect data rows until a fully empty row or next year header
    data: List[List[object]] = []
    r = header_row + 1
    year_header_re = re.compile(r"a[ñn]o\s*\d{4}", re.IGNORECASE)
    while r < len(df):
        raw = df.iloc[r].tolist()
        # stop if we encounter next year header like 'Año 2024'
        if any((not pd.isna(x)) and year_header_re.search(str(x)) for x in raw):
            break
        if all(pd.isna(x) or str(x).strip() == "" for x in raw):
            break
        data.append(raw)
        r += 1

    if not data:
        return pd.DataFrame(columns=["Mes","IV"])

    headers = headers[:len(data[0])]
    tbl = pd.DataFrame(data, columns=headers)

    # Select columns by label proximity, handling duplicates
    candidates = []
    for idx, col in enumerate(tbl.columns):
        cl = (col or "").lower()
        if "mes" in cl:
            candidates.append(("Mes", idx, col))
        if any(k in cl for k in ["iv", "índice de viviendas", "indice de viviendas", "indice viviendas"]):
            candidates.append(("IV", idx, col))

    if not candidates:
        # fallback heuristic: first col as Mes, most numeric as IV
        mes_col_name = tbl.columns[0]
        numeric_scores = [(c, pd.to_numeric(tbl[c], errors="coerce").notna().sum()) for c in tbl.columns[1:]]
        iv_col_name = max(numeric_scores, key=lambda x: x[1])[0] if numeric_scores else (tbl.columns[1] if tbl.shape[1] > 1 else tbl.columns[0])
        out = tbl[[mes_col_name, iv_col_name]].copy()
        out.columns = ["Mes","IV"]
        out["Mes"] = out["Mes"].map(normalize_month)
        out = out[out["Mes"].notna()].copy()
        out["IV"] = pd.to_numeric(out["IV"], errors="coerce")
        return out.dropna(subset=["IV"])        

    def choose_by_proximity(kind: str) -> Optional[str]:
        items = [(abs(idx - anchor_col), idx, label) for k, idx, label in candidates if k == kind] if anchor_col is not None else [(idx, idx, label) for k, idx, label in candidates if k == kind]
        if not items:
            return None
        items.sort()
        return items[0][2]

    mes_pick = choose_by_proximity("Mes")
    iv_pick = choose_by_proximity("IV")

    # Final fallback if any missing
    if mes_pick is None:
        mes_pick = next((c for c in tbl.columns if "mes" in c.lower()), tbl.columns[0])
    if iv_pick is None:
        numeric_scores = [(c, pd.to_numeric(tbl[c], errors="coerce").notna().sum()) for c in tbl.columns]
        iv_pick = max(numeric_scores, key=lambda x: x[1])[0]

    out = tbl[[mes_pick, iv_pick]].copy()
    out.columns = ["Mes","IV"]
    out["Mes"] = out["Mes"].map(normalize_month)
    out = out[out["Mes"].notna()].copy()
    out["IV"] = pd.to_numeric(out["IV"], errors="coerce")
    out = out.dropna(subset=["IV"])  # keep only numeric IV rows
    return out


def read_sheet_as_raw(sheet_name: str) -> pd.DataFrame:
    # Read with no header to preserve layout of messy tables
    return pd.read_excel(excel_path, sheet_name=sheet_name, header=None, engine="openpyxl")



In [14]:
# Extract OASIS and LA QUININA (single neighborhood per sheet)

def extract_simple_sheet(sheet_name: str, neighborhood: str) -> pd.DataFrame:
    df_raw = read_sheet_as_raw(sheet_name)
    year_hits = find_year_cells(df_raw, year="2023")
    if not year_hits:
        return pd.DataFrame(columns=["Neighborhood","Mes","IV"])
    # Assume first match is the target block
    (start_row, anchor_col) = year_hits[0]
    table = extract_table_below(df_raw, start_row=start_row, expected_cols=["Mes","IV"], anchor_col=anchor_col)
    if table.empty:
        return pd.DataFrame(columns=["Neighborhood","Mes","IV"])
    table.insert(0, "Neighborhood", neighborhood)
    return table

iv_oasis = extract_simple_sheet("OASIS", "OASIS")
iv_laquinina = extract_simple_sheet("LA QUININA", "LA QUININA")

iv_oasis.head(), iv_laquinina.head()


(  Neighborhood      Mes     IV
 0        OASIS    Enero  30.65
 1        OASIS  Febrero  29.00
 2        OASIS    Marzo  29.45
 3        OASIS    Abril  28.30
 4        OASIS     Mayo  27.12,
   Neighborhood      Mes     IV
 0   LA QUININA    Enero  29.25
 1   LA QUININA  Febrero  27.26
 2   LA QUININA    Marzo  29.00
 3   LA QUININA    Abril  26.33
 4   LA QUININA     Mayo  26.25)

In [15]:
# Extract from multi-neighborhood sheets with headers in second row and blocks per year below

def extract_multi_sheet(sheet_name: str, neighborhood_headers: List[str]) -> pd.DataFrame:
    df_raw = read_sheet_as_raw(sheet_name)
    if df_raw.empty:
        return pd.DataFrame(columns=["Neighborhood","Mes","IV"])

    # Identify the header row (2nd row, index 1) containing neighborhood names
    header_row_idx = 1 if len(df_raw) > 1 else 0
    header_row = [normalize_header(x) for x in df_raw.iloc[header_row_idx].tolist()]

    # Map each neighborhood header to a column index where it appears
    header_positions: Dict[str, int] = {}
    for idx, val in enumerate(header_row):
        low = val.lower()
        for nh in neighborhood_headers:
            if nh.lower() in low and nh not in header_positions:
                header_positions[nh] = idx

    # Work with a slice of the dataframe starting a couple of rows below header
    sub = df_raw.iloc[header_row_idx+1:].reset_index(drop=True)

    # Find all year markers once
    year_hits = find_year_cells(sub, year="2023")

    # For each neighborhood, pick the year marker closest in column to its header
    all_parts: List[pd.DataFrame] = []
    for nh in neighborhood_headers:
        nh_col = header_positions.get(nh)
        if nh_col is None:
            continue
        # choose closest year hit by column distance
        if not year_hits:
            continue
        yr_row, yr_col = sorted(year_hits, key=lambda rc: abs(rc[1] - nh_col))[0]
        tbl = extract_table_below(sub, start_row=yr_row, expected_cols=["Mes","IV"], anchor_col=nh_col)
        if tbl.empty:
            continue
        tbl.insert(0, "Neighborhood", nh)
        all_parts.append(tbl)

    if all_parts:
        return pd.concat(all_parts, ignore_index=True)
    return pd.DataFrame(columns=["Neighborhood","Mes","IV"])

# OASIS-B neighborhoods
iv_oasis_b = extract_multi_sheet("OASIS-B", ["ONDAS DEL CARIBE", "PANTANO", "8 DE DICIEMBRE"])

# LA QUININA-B neighborhoods
iv_laquinina_b = extract_multi_sheet("LA QUININA-B", ["LA COQUERA", "SAN JACINTO", "NUEVA BETHEL"])

iv_oasis_b.head(), iv_laquinina_b.head()


(       Neighborhood      Mes    IV
 0  ONDAS DEL CARIBE    Enero  27.8
 1  ONDAS DEL CARIBE  Febrero  30.3
 2  ONDAS DEL CARIBE    Marzo  28.2
 3  ONDAS DEL CARIBE    Abril  29.4
 4  ONDAS DEL CARIBE     Mayo  25.9,
   Neighborhood      Mes    IV
 0   LA COQUERA    Enero  30.4
 1   LA COQUERA  Febrero  28.5
 2   LA COQUERA    Marzo  27.1
 3   LA COQUERA    Abril  24.8
 4   LA COQUERA     Mayo  29.2)

In [17]:
# Assemble unified DataFrame
parts = []
for part in [iv_oasis, iv_laquinina, iv_oasis_b, iv_laquinina_b]:
    if part is not None and not part.empty:
        parts.append(part)

df_iv_2023 = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=["Neighborhood","Mes","IV"]) 

# Normalize ordering of months and neighborhood names
if not df_iv_2023.empty:
    df_iv_2023["Neighborhood"] = df_iv_2023["Neighborhood"].str.strip()
    df_iv_2023["MesOrder"] = df_iv_2023["Mes"].map(MONTH_ORDER)
    df_iv_2023 = df_iv_2023.sort_values(["Neighborhood","MesOrder"]).drop(columns=["MesOrder"]) 

df_iv_2023.head(50)


,Neighborhood,Mes,IV
48,8 DE DICIEMBRE,Enero,30.800000
49,8 DE DICIEMBRE,Febrero,35.000000
50,8 DE DICIEMBRE,Marzo,34.500000
51,8 DE DICIEMBRE,Abril,33.200000
52,8 DE DICIEMBRE,Mayo,30.800000
53,8 DE DICIEMBRE,Junio,33.000000
54,8 DE DICIEMBRE,Julio,31.900000
55,8 DE DICIEMBRE,Agosto,34.000000
56,8 DE DICIEMBRE,Septiembre,30.900000
57,8 DE DICIEMBRE,Octubre,29.800000


In [18]:
# Basic sanity checks / preview
print("Rows:", len(df_iv_2023))
print("Neighborhoods:", sorted(df_iv_2023["Neighborhood"].unique().tolist()) if not df_iv_2023.empty else [])

# Show a pivot preview: months vs neighborhood
if not df_iv_2023.empty:
    pivot = df_iv_2023.pivot_table(index="Mes", columns="Neighborhood", values="IV", aggfunc="mean")
    display(pivot)

# Keep df_iv_2023 in memory for later use


Rows: 96
Neighborhoods: ['8 DE DICIEMBRE', 'LA COQUERA', 'LA QUININA', 'NUEVA BETHEL', 'OASIS', 'ONDAS DEL CARIBE', 'PANTANO', 'SAN JACINTO']


Neighborhood,8 DE DICIEMBRE,LA COQUERA,LA QUININA,NUEVA BETHEL,OASIS,ONDAS DEL CARIBE,PANTANO,SAN JACINTO
Mes,,,,,,,,
Abril,33.2,24.8,26.330000,28.4,28.300000,29.4,23.8,22.8
Agosto,34.0,30.1,16.666667,30.5,26.666667,30.1,24.1,22.9
Diciembre,34.0,27.3,13.157895,27.2,13.333333,30.8,23.9,27.8
Enero,30.8,30.4,29.250000,27.5,30.650000,27.8,24.5,25.7
Febrero,35.0,28.5,27.260000,29.3,29.000000,30.3,26.7,27.3
Julio,31.9,25.9,25.120000,24.6,22.608696,29.7,22.8,24.2
Junio,33.0,26.7,25.450000,26.0,28.140000,27.2,25.9,26.9
Marzo,34.5,27.1,29.000000,23.1,29.450000,28.2,28.9,23.5
Mayo,30.8,29.2,26.250000,31.2,27.120000,25.9,27.1,28.2


In [19]:
# If df_iv_2023 is long-form, pivot to wide (months as rows, neighborhoods as columns)
if set(["Neighborhood", "Mes", "IV"]).issubset(df_iv_2023.columns):
    wide = df_iv_2023.pivot_table(index="Mes", columns="Neighborhood", values="IV", aggfunc="mean")
else:
    wide = df_iv_2023.copy()

# Canonical neighborhood → id mapping
neighborhood_to_id = {
    "OASIS": 1,
    "LA QUININA": 2,
    "ONDAS DEL CARIBE": 3,
    "PANTANO": 4,
    "8 DE DICIEMBRE": 5,
    "LA COQUERA": 6,
    "SAN JACINTO": 7,
    "NUEVA BETHEL": 8,
}
# Validate names: error if any neighborhood in the data is not in the mapping
unknown_cols = sorted(set(wide.columns) - set(neighborhood_to_id.keys()))
if unknown_cols:
    raise ValueError(f"Neighborhood(s) not in mapping: {unknown_cols}")

# Ensure all required neighborhoods are present
ordered = [name for name, nid in sorted(neighborhood_to_id.items(), key=lambda kv: kv[1])]
missing_required = [name for name in ordered if name not in wide.columns]
if missing_required:
    raise ValueError(f"Missing required neighborhood column(s) in df_iv_2023: {missing_required}")

# Convert percent → fraction
wide_frac = wide[ordered] / 100.0

# Annual mean per neighborhood across months
means = wide_frac.mean(axis=0, skipna=True)

# Build output in required schema and order
out = pd.DataFrame({
    "neighborhood_id": [neighborhood_to_id[name] for name in ordered],
    "neighborhood_name": ordered,
    "hi_mean": [means.loc[name] for name in ordered],
})

# Write CSV exactly as requested
out.to_csv("hi_by_neighborhood_2023.csv", index=False)